# Esteira Geo — Processamento Interativo

Fluxo idêntico ao executado pelo watcher: **Bronze → Silver → Gold → PostGIS**

Variáveis de ambiente já injetadas pelo Docker (MinIO + PostGIS da rede interna).

## 0. Configuração

In [1]:
import os, sys
sys.path.insert(0, '/app')  # módulos da esteira

# Altere aqui para processar outro use_case
os.environ['USE_CASE'] = 'enchentes_poa'

import importlib, config
importlib.reload(config)  # recarrega config com o USE_CASE atualizado

print(f"Use case : {config.USE_CASE}")
print(f"Bronze   : s3://{config.AWS_S3_BRONZE_BUCKET}/{config.S3_BRONZE_PREFIX}")
print(f"Silver   : s3://{config.AWS_S3_SILVER_BUCKET}/{config.S3_SILVER_PREFIX}")
print(f"Gold     : s3://{config.AWS_S3_GOLD_BUCKET}/{config.S3_GOLD_PREFIX}")
print(f"PostGIS  : {config.RDS_HOST}:{config.RDS_PORT}/{config.RDS_DATABASE}")

✓ Configuration loaded (Mode: MINIO)
  Storage: MINIO
    MinIO: http://minio:9000
    Buckets: bronze/enchentes_poa, silver/enchentes_poa, gold/enchentes_poa
  Use Case: enchentes_poa
  Database: postgis:5432/esteira_geo
  Logging: logs/pipeline.log
✓ Configuration loaded (Mode: MINIO)
  Storage: MINIO
    MinIO: http://minio:9000
    Buckets: bronze/enchentes_poa, silver/enchentes_poa, gold/enchentes_poa
  Use Case: enchentes_poa
  Database: postgis:5432/esteira_geo
  Logging: logs/pipeline.log
Use case : enchentes_poa
Bronze   : s3://bronze/enchentes_poa/
Silver   : s3://silver/enchentes_poa/
Gold     : s3://gold/enchentes_poa/
PostGIS  : postgis:5432/esteira_geo


## 1. Inspecionar Bronze (S3)

In [2]:
import boto3

s3 = boto3.client(
    's3',
    endpoint_url=config.AWS_ENDPOINT_URL,
    aws_access_key_id=config.AWS_ACCESS_KEY_ID,
    aws_secret_access_key=config.AWS_SECRET_ACCESS_KEY,
    region_name=config.AWS_S3_REGION_NAME,
)

resp = s3.list_objects_v2(Bucket=config.AWS_S3_BRONZE_BUCKET, Prefix=config.S3_BRONZE_PREFIX)
bronze_files = [o['Key'] for o in resp.get('Contents', [])]
print(f"{len(bronze_files)} arquivo(s) no bronze:")
for f in bronze_files:
    print(f"  {f}")

/usr/local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


2 arquivo(s) no bronze:
  enchentes_poa/processados/citizens_data.geojson
  enchentes_poa/processados/flooding_areas_porto_alegre.geojson


## 2. Silver — Normalização

In [4]:
from etl.silver_processor import process_silver

silver = process_silver()

if 'flooding' in silver:
    print(f"Áreas de enchente: {len(silver['flooding'])} registros")
    display(silver['flooding'].head())

if 'citizens' in silver:
    print(f"\nCidadãos: {len(silver['citizens'])} registros")
    display(silver['citizens'].head())

RuntimeError: Nenhum arquivo encontrado no bucket bronze para processar.

## 3. Gold — Batimento Geográfico (Spatial Join)

In [ ]:
from etl.gold_processor import process_gold, silver_ready

has_areas, has_citizens = silver_ready()
print(f"Silver pronto — áreas: {has_areas} | cidadãos: {has_citizens}")

if has_areas and has_citizens:
    affected, unaffected, all_citizens = process_gold()

    total = len(all_citizens)
    print(f"\nResultado do batimento:")
    print(f"  Atingidos    : {len(affected)} ({len(affected)/total*100:.1f}%)")
    print(f"  Não atingidos: {len(unaffected)}")
    print(f"  Total        : {total}")

    display(affected.head())
else:
    print("Silver incompleto — execute as células anteriores primeiro.")

## 4. PostGIS — Sincronização

In [ ]:
from etl.postgis_loader import load_to_postgis

load_to_postgis(sync_areas=has_areas, sync_citizens=(has_areas and has_citizens))
print("PostGIS sincronizado.")

## 5. Consultas no PostGIS

In [ ]:
import psycopg2, pandas as pd

conn = psycopg2.connect(
    host=config.RDS_HOST, port=config.RDS_PORT,
    dbname=config.RDS_DATABASE, user=config.RDS_USER, password=config.RDS_PASSWORD
)

use_case = config.USE_CASE

stats = pd.read_sql(f"""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN affected_by_flooding THEN 1 ELSE 0 END) AS atingidos,
        SUM(CASE WHEN NOT affected_by_flooding THEN 1 ELSE 0 END) AS nao_atingidos
    FROM {use_case}_citizens
""", conn)

display(stats)

sample = pd.read_sql(f"""
    SELECT citizen_id, name, affected_by_flooding, ST_AsText(geometry) AS geom
    FROM {use_case}_citizens
    WHERE affected_by_flooding = TRUE
    LIMIT 5
""", conn)

display(sample)
conn.close()

## 6. Visualização no Mapa (Leaflet via IFrame)

In [ ]:
from IPython.display import IFrame

# Abre o mapa do Flask (certifique-se que o serviço web está rodando)
IFrame(src=f'http://localhost:5000/map?use_case={config.USE_CASE}', width='100%', height=600)

## 7. Ingestão Manual de Novos Dados

Faça upload de um CSV ou GeoJSON direto para o bronze e reprocesse sem precisar do watcher.

In [ ]:
# Exemplo: upload de um CSV local para o bronze
local_file = '/data/bronze/enchentes_poa/citizens_sample.csv'  # ajuste o caminho
s3_key = f"{config.USE_CASE}/{os.path.basename(local_file)}"

s3.upload_file(local_file, config.AWS_S3_BRONZE_BUCKET, s3_key)
print(f"Upload: s3://{config.AWS_S3_BRONZE_BUCKET}/{s3_key}")

# Reprocessar (equivalente ao watcher disparando main.py)
silver = process_silver()
has_areas, has_citizens = silver_ready()
if has_areas and has_citizens:
    affected, unaffected, all_citizens = process_gold()
    load_to_postgis(sync_areas=True, sync_citizens=True)
    print(f"Reprocessado: {len(affected)} atingidos / {len(all_citizens)} total")